# Exercise 1 (3 points)

Which of the following methods is considered a non-stochastic method in Prescriptive Analytics?

- (a) Monte Carlo Simulation
- (b) Decision Trees
- (c) Markov Decision Process
- (d) Random Walk Model
- (e) None of the above

B

# Exercise 2 (4 points)

Briefly define a decision tree and describe its strength in Prescriptive Analytics. Be specific.

A decision tree is a structured tree model for decision-making that maps out possible outcomes, actions, and consequences. In prescriptive their strength comes from their ability to model sequental decisions

# Exercise 3 (3 points)

Which of the following best describes the term *stochastic method* in Prescriptive Analytics?

- (a) It ignores randomness and relies on logic-based rules.
- (b) It explicitly incorporates and uses random variations.
- (c) It is always used in strategic decision making only.
- (d) It only applies to supervised machine learning models.
- (e) All of the above.

B

# Exercise 4 (4 points)

Describe the importance of simulations in complex system analysis. Why might simulations be preferred over direct experimentation in such systems?

they offer insight into different outcomes when real-world experimentation is impractical or impossible.

# Exercise 5 

Consider the `ai_dev_productivity.csv` data file. This dataset simulates the productivity of AI developers over 500 days, capturing the subtle interaction between deep work, distractions, caffeine intake, and code quality. Designed to push the limits of machine learning, this data blends behavioral, physiological, and productivity indicators to allow for advanced predictive modeling. This is brief description of each of the columns in the data:

- `hours_coding`: Total focused hours spent on software development work (0–12 hours).
- `coffee_intake_mg`: Daily caffeine intake in milligrams (0–600 mg).
- `distractions`: Number of distractions (e.g., meetings, Slack notifications) (0–10).
- `sleep_hours`: Number of hours of sleep the previous night (3–10 hours).
- `commits`: Number of code commits pushed during the day (0–20).
- `bugs_reported`: Number of bugs reported in code written that day (0–10).
- `ai_usage_hours`: Number of hours spent using AI tools (e.g., ChatGPT, Copilot) (0–12).
- `cognitive_load`: Self-reported mental strain on a scale of 1 to 10. 
- `task_success`: Target column — whether the daily productivity goal was achieved (0/1).

## Exercise 5(a) (2 points)

Read the `ai_dev_productivity.csv` data-file and create a data-frame called `df`.

In [26]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score
from sklearn.tree import DecisionTreeClassifier

df = pd.read_csv('ai_dev_productivity.csv')

## Exercise 5(b) (2 points)

Report the relative frequency of `task_success`.

In [27]:
df['task_success'].value_counts(normalize=True)

task_success
1    0.606
0    0.394
Name: proportion, dtype: float64

## Exercise 5(c) (5 points)

Use `task_success` as the target feature and the other features as the inputs. Split the data into `train (80%)` and `test (20%)`. 

In [28]:
X = df.drop(columns=['task_success'])
y = df['task_success']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Exercise 5(d) (8 points)

Train and evaluate the `LogistRegression` model over 5-folds using the only the `train` dataset. Evalute the `recall` to evaluate model performance. Use 15% as threshold. 

In [29]:
scores = []

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for  i, (train_index, test_index) in enumerate(skf.split(X_train, y_train)):
    X_train_fold, X_test_fold = X_train.iloc[train_index], X_train.iloc[test_index]
    y_train_fold, y_test_fold = y_train.iloc[train_index], y_train.iloc[test_index]

    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_train_fold, y_train_fold)

    y_pred = model.predict_proba(X_test_fold)[:, 1]
    y_label = (y_pred >= 0.15).astype(int)
    recall = recall_score(y_test_fold, y_label)
    scores.append(recall)

    print(f'Recall for fold {i+1}: {recall:.4f}')
    
print('Average recall across folds:', np.mean([scores]))

Recall for fold 1: 1.0000
Recall for fold 2: 1.0000
Recall for fold 3: 1.0000
Recall for fold 4: 0.9787
Recall for fold 5: 0.9574
Average recall across folds: 0.9872340425531915


## Exercise 5(e) (8 points)

Train and evaluate the `DecisionTreeClassifier` model (use `max_depth=3`, `random_state=10`) over 5-folds using the only the `train` dataset. Evalute the `recall` to evaluate model performance. Use 15% as threshold. 

In [30]:
scores = []

for  i, (train_index, test_index) in enumerate(skf.split(X_train, y_train)):
    X_train_fold, X_test_fold = X_train.iloc[train_index], X_train.iloc[test_index]
    y_train_fold, y_test_fold = y_train.iloc[train_index], y_train.iloc[test_index]
    
    model = DecisionTreeClassifier(max_depth=3, random_state=10)
    model.fit(X_train_fold, y_train_fold)
    
    y_pred = model.predict_proba(X_test_fold)[:,1]
    y_label = (y_pred >= 0.15).astype(int)
    recall = recall_score(y_test_fold, y_label)
    scores.append(recall)
    
    print(f'Fold {i+1}, Recall: {recall:.4f}')

print(f'Average recall over 5 folds: {np.mean(scores):.4f}')

Fold 1, Recall: 1.0000
Fold 2, Recall: 1.0000
Fold 3, Recall: 1.0000
Fold 4, Recall: 1.0000
Fold 5, Recall: 1.0000
Average recall over 5 folds: 1.0000


## Exercise 5(f) (2 points)

Based on the `recall`, what model would you use to predict `task_success`?

Based on my results I would use the decision tree classifier

## Exercise 5(g) (7 points)

Based on your answer from part 4(f), use that model to predict on the likelihood of `task_success` on the `test` dataset.

In [31]:
test_pred = []

for  i, (train_index, test_index) in enumerate(skf.split(X_train, y_train)):
    X_train_fold, X_test_fold = X_train.iloc[train_index], X_train.iloc[test_index]
    y_train_fold, y_test_fold = y_train.iloc[train_index], y_train.iloc[test_index]
    
    model = DecisionTreeClassifier(max_depth=3, random_state=10)
    model.fit(X_train_fold, y_train_fold)
    
    y_pred = model.predict_proba(X_test)[:,1]
    test_pred.append(y_pred)
    
test = pd.concat([X_test.reset_index(drop=True),y_test.reset_index(drop=True)], axis=1)
test['predicted_attrition'] = np.mean(test_pred, axis=0)
test = test.sort_values(by='predicted_attrition', ascending=False).reset_index(drop=True)
test.head(10)

,hours_coding,coffee_intake_mg,distractions,sleep_hours,commits,bugs_reported,ai_usage_hours,cognitive_load,task_success,predicted_attrition
0,4.93,509,1,5.2,3,0,0.30,4.5,1,1.0
1,5.99,600,1,5.8,2,1,0.71,5.4,1,1.0
2,5.14,590,3,7.2,6,0,2.35,3.6,1,1.0
3,4.85,467,3,7.9,4,1,1.26,3.6,1,1.0
4,8.13,600,2,5.7,9,0,1.69,4.6,1,1.0
5,5.14,480,2,8.0,2,3,1.97,2.5,1,1.0
6,6.19,600,5,8.9,10,1,2.47,3.7,1,1.0
7,5.19,523,3,7.1,7,0,1.78,6.2,1,1.0
8,4.40,479,5,4.9,0,0,0.19,6.9,1,1.0
9,5.62,441,3,7.5,5,0,0.16,4.6,1,1.0
